# Инференс модели из MLflow Registry (тег PRD)

Чистый ноутбук для загрузки финальной модели с алиасом `prd` из MLflow Model Registry и выполнения тестового предикта.

## 1. Импорты

In [ ]:
import os
import pandas as pd
import mlflow
import mlflow.pyfunc
from mlflow.tracking import MlflowClient
from dotenv import load_dotenv

load_dotenv()

## 2. Подключение к MLflow и MinIO S3

In [ ]:
os.environ["AWS_ACCESS_KEY_ID"] = os.getenv("AWS_ACCESS_KEY_ID", "admin")
os.environ["AWS_SECRET_ACCESS_KEY"] = os.getenv("AWS_SECRET_ACCESS_KEY", "password")

raw_s3_endpoint = os.getenv("MLFLOW_S3_ENDPOINT_URL", "http://localhost:9000")
if "minio:9000" in raw_s3_endpoint:
    raw_s3_endpoint = "http://localhost:9000"
os.environ["MLFLOW_S3_ENDPOINT_URL"] = raw_s3_endpoint

MLFLOW_TRACKING_URI = os.getenv("MLFLOW_TRACKING_URI", "http://localhost:5050")
mlflow.set_tracking_uri(MLFLOW_TRACKING_URI)

mlflow.search_experiments(max_results=1)
print(f"Подключение к MLflow успешно: {mlflow.get_tracking_uri()}")

## 3. Просмотр информации о зарегистрированной модели

In [ ]:
REGISTERED_MODEL_NAME = "catboost_optimized"

client = MlflowClient()
model_version = client.get_model_version_by_alias(REGISTERED_MODEL_NAME, "prd")

print(f"Модель:  {REGISTERED_MODEL_NAME}")
print(f"Версия:  {model_version.version}")
print(f"Алиас:   prd")
print(f"Теги:    {model_version.tags}")
print(f"Run ID:  {model_version.run_id}")
print(f"Source:  {model_version.source}")

## 4. Загрузка модели из реестра по алиасу `prd`

In [ ]:
model_uri = f"models:/{REGISTERED_MODEL_NAME}@prd"
model = mlflow.pyfunc.load_model(model_uri)

print(f"Модель загружена: {model_uri}")
print(f"Тип модели: {type(model)}")

## 5. Подготовка тестовых данных

In [ ]:
df_test = pd.read_csv("../data/test_with_new_features.csv")
df_test = df_test.drop(columns=["Unnamed: 0", "id", "address", "address_rus"])
df_test = df_test.dropna()
df_test["population"] = df_test["population"].astype(str).str.replace("\xa0", "").astype(float)
df_test["atm_group"] = df_test["atm_group"].astype(float)

int_cols = [
    "schools_nearby", "supermarket_nearby", "mall_nearby", "bar_nearby",
    "cafe_nearby", "restaurant_nearby", "police_nearby", "post_office_nearby",
    "place_of_worship_nearby", "university_nearby", "cinema_nearby",
    "casino_nearby", "nightclub_nearby"
]
float_cols = ["atm_group", "lat", "long", "atm_nearby", "population"]

X_test = df_test.drop(columns=["target"])
y_test = df_test["target"]

X_test[int_cols] = X_test[int_cols].astype("int64")
X_test[float_cols] = X_test[float_cols].astype("float64")

print(f"Тестовая выборка: {X_test.shape[0]} строк, {X_test.shape[1]} признаков")
X_test.head(3)

## 6. Тестовый предикт

In [ ]:
sample = X_test.head(5).copy()
predictions = model.predict(sample)

result = sample.copy()
result["predicted_target"] = predictions
result["actual_target"] = y_test.head(5).values
result["abs_error"] = (result["predicted_target"] - result["actual_target"]).abs()

print("Предсказания PRD-модели на 5 примерах:")
print(result[["predicted_target", "actual_target", "abs_error"]].to_string())

## 7. Предикт на всей тестовой выборке

In [ ]:
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error

all_predictions = model.predict(X_test)

r2 = r2_score(y_test, all_predictions)
mae = mean_absolute_error(y_test, all_predictions)
mse = mean_squared_error(y_test, all_predictions)

print("Метрики PRD-модели на тестовой выборке:")
print(f"  R²:  {r2:.4f}")
print(f"  MAE: {mae:.4f}")
print(f"  MSE: {mse:.6f}")